# Les files — Exercice 2 : `passe_ton_tour`, `nb_occurences`, `last_is_first`

**Notions travaillées :** utiliser une structure **uniquement par son interface** (ses méthodes), parcourir une file sans la modifier (rotation).

## Mise en place

On importe la classe `File` écrite à l'exercice 1 (le fichier `file.py` doit être **dans le même dossier** que ce notebook).

L'énoncé interdit d'utiliser le type `list` : on ne manipule les files **qu'avec leurs méthodes** `enfile`, `defile`, `taille`, `est_vide`. La petite fonction `file_de` sert uniquement à fabriquer rapidement des files pour les tests.

On affiche une file sous la forme `tête < ... < queue` : on **défile à gauche** et on **enfile à droite**.

In [1]:
from file import File   # la classe File de l'exercice 1 (fichier file.py placé à côté du notebook)


def file_de(*elements):
    """Outil pour les tests : construit une file contenant les éléments donnés (le premier en tête)."""
    f = File()
    for el in elements:
        f.enfile(el)
    return f


print(file_de("a", "b", "c"))

tête < 'a' < 'b' < 'c' < queue


## Question 1 : `passe_ton_tour(f)`

On veut placer **en queue** l'élément qui est **en tête**. C'est exactement ce que font, à la suite, `defile` (qui retire la tête) puis `enfile` (qui ajoute en queue) :
```python
f.enfile(f.defile())
```

⚠️ **Cas particulier :** si la file est vide, `defile` déclenche une erreur. On choisit de **ne rien faire** dans ce cas (passer son tour dans une file vide n'a pas de sens, mais ce n'est pas une raison pour planter).

💡 La fonction **ne renvoie rien** : elle **modifie** la file passée en argument. Une file est un objet : la fonction reçoit le **même objet** que l'appelant, pas une copie.

In [2]:
def passe_ton_tour(f):
    if not f.est_vide():
        f.enfile(f.defile())


f = file_de(1, 2, 3, 4)
passe_ton_tour(f)
print(f)
assert repr(f) == repr(file_de(2, 3, 4, 1))

g = File()
passe_ton_tour(g)        # file vide : pas d'erreur
assert g.est_vide()

tête < 2 < 3 < 4 < 1 < queue


## Question 2 : `nb_occurences(f, val)`

(L'énoncé écrit *occurences* ; l'orthographe correcte est *occu**rr**ences*, mais on garde le nom demandé.)

**Difficulté :** pour compter, il faut regarder **tous** les éléments, or on ne voit que la tête. Et la file doit être **identique** après l'appel.

> **🔑 La technique clé de tout le TD : la rotation**
>
> Avec une file, on ne voit que la **tête**. Pour regarder tous les éléments, on fait faire **un tour complet** à la file :
> on défile un élément, on le traite, puis on le **ré-enfile**. Après exactement `f.taille()` rotations, chaque élément est revenu à sa place : la file est **identique** à ce qu'elle était au départ.
>
> ```python
> for _ in range(f.taille()):
>     el = f.defile()
>     ...            # traiter el
>     f.enfile(el)
> ```
>
> ⚠️ On calcule `f.taille()` **une seule fois**, au départ (c'est ce que fait `range`). Une boucle `while not f.est_vide()` ne s'arrêterait jamais, puisqu'on ré-enfile chaque élément.


**Algorithme :** on fait un tour complet ; à chaque élément défilé, on augmente un compteur s'il est égal à `val`, puis on le ré-enfile.

In [3]:
def nb_occurences(f, val):
    compteur = 0
    for _ in range(f.taille()):
        el = f.defile()
        if el == val:
            compteur += 1
        f.enfile(el)              # on remet l'élément : la file fait un tour complet
    return compteur


f = file_de(3, 1, 3, 2, 3)
print(nb_occurences(f, 3), "fois le nombre 3")
print("La file après l'appel :", f)

assert nb_occurences(f, 3) == 3
assert nb_occurences(f, 7) == 0
assert repr(f) == repr(file_de(3, 1, 3, 2, 3))     # file inchangée
assert nb_occurences(File(), 1) == 0

3 fois le nombre 3
La file après l'appel : tête < 3 < 1 < 3 < 2 < 3 < queue


⚠️ **Erreur fréquente :** compter avec `while not f.est_vide(): ... f.defile()` **sans ré-enfiler** : le compte est juste, mais la file est **vidée**. C'est justement ce que l'énoncé interdit.

## Question 3 : `last_is_first(f)`

On veut que le **dernier** élément passe **en premier**, sans changer l'ordre des autres :
`a b c d` → `d a b c`.

**Idée :** il suffit de faire passer leur tour à **tous les éléments sauf le dernier**. Si la file contient `n` éléments, on fait `n - 1` rotations :

| Rotation | File |
|:---|:---|
| départ | `a b c d` |
| 1 | `b c d a` |
| 2 | `c d a b` |
| 3 | `d a b c` ✔ |

On réutilise la fonction `passe_ton_tour` de la question 1 : c'est l'intérêt d'écrire de petites fonctions.

💡 Si la file est vide, `range(-1)` ne fait aucun tour : pas de cas particulier à traiter.

In [4]:
def last_is_first(f):
    for _ in range(f.taille() - 1):
        passe_ton_tour(f)


f = file_de("a", "b", "c", "d")
last_is_first(f)
print(f)
assert repr(f) == repr(file_de("d", "a", "b", "c"))

for g, attendu in [(File(), File()), (file_de(1), file_de(1)), (file_de(1, 2), file_de(2, 1))]:
    last_is_first(g)
    assert repr(g) == repr(attendu)
print("Tous les tests passent ✔")

tête < 'd' < 'a' < 'b' < 'c' < queue
Tous les tests passent ✔
